# SDNET2018 data audit and manifest

This notebook downloads or reuses the official SDNET2018 archive, verifies it before extraction, audits every image, records exact duplicates without deleting source files, and writes portable CSV manifests.

Parent-photo IDs are never guessed. If no verified parent mapping is supplied, the research split remains unset and the optional preliminary split is labelled tile_development_only.


## 1. Dataset source and expected structure

Official record: https://digitalcommons.usu.edu/all_datasets/48/

Expected folders: D/CD, D/UD, P/CP, P/UP, W/CW, and W/UW. Labels use crack=1 and non-crack=0 so the positive class matches the later weighted BCE loss.


In [ ]:
from collections import Counter, defaultdict
import csv
import hashlib
import json
import os
import shutil
from pathlib import Path
import random
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
from zipfile import ZipFile

from PIL import Image

SEED = 42
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/drive/MyDrive/Fridge/major_project_dataset")
DATA_DIR = PROJECT_ROOT
ARCHIVE_PATH = DATA_DIR / "DATA_Maguire_20180517_ALL.zip"
EXTRACT_DIR = DATA_DIR / "SDNET2018"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_audit"

OFFICIAL_RECORD_URL = "https://digitalcommons.usu.edu/all_datasets/48/"
ARCHIVE_URL = (
    "https://digitalcommons.usu.edu/cgi/viewcontent.cgi?"
    "article=1047&context=all_datasets&filename=2&type=additional"
)
EXPECTED_ARCHIVE_MD5 = "677411e784f194422c90f52d9ed0d7c6"

CLASS_INFO = {
    "CD": {"label": 1, "class_name": "crack", "surface": "deck"},
    "UD": {"label": 0, "class_name": "non_crack", "surface": "deck"},
    "CP": {"label": 1, "class_name": "crack", "surface": "pavement"},
    "UP": {"label": 0, "class_name": "non_crack", "surface": "pavement"},
    "CW": {"label": 1, "class_name": "crack", "surface": "wall"},
    "UW": {"label": 0, "class_name": "non_crack", "surface": "wall"},
}
EXPECTED_COUNTS = {
    "CD": 2025,
    "UD": 11595,
    "CP": 2608,
    "UP": 21726,
    "CW": 3851,
    "UW": 14287,
}
IMAGE_SUFFIXES = {".jpg", ".jpeg"}

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "work_dir": str(PROJECT_ROOT.resolve()),
    "archive": str(ARCHIVE_PATH.resolve()),
    "official_record": OFFICIAL_RECORD_URL,
})

def save_json(path, value):
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as stream:
        json.dump(value, stream, indent=2)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)


## 2. Download the official archive

The file is roughly 516 MB. Existing downloads are reused. If the host blocks automated access, download SDNET2018.zip from the official record and place it at the archive path printed above.


In [ ]:
def download_file(url, destination, chunk_size=1024 * 1024):
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urlopen(request, timeout=60) as response, destination.open("wb") as output:
            total = int(response.headers.get("Content-Length", 0))
            downloaded = 0
            while chunk := response.read(chunk_size):
                output.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(
                        f"\rDownloaded {downloaded / 1024**2:.1f} / "
                        f"{total / 1024**2:.1f} MiB",
                        end="",
                    )
        print()
    except (HTTPError, URLError) as error:
        destination.unlink(missing_ok=True)
        raise RuntimeError(
            f"Automatic download failed: {error}. Download the archive from "
            f"{OFFICIAL_RECORD_URL} and place it at {destination.resolve()}."
        ) from error

if ARCHIVE_PATH.exists():
    print(f"Reusing {ARCHIVE_PATH.resolve()}")
else:
    download_file(ARCHIVE_URL, ARCHIVE_PATH)

print(f"Archive size: {ARCHIVE_PATH.stat().st_size / 1024**2:.1f} MiB")


## 3. Verify archive integrity

The official record publishes a 32-character checksum, treated here as MD5. A SHA-256 digest is also recorded for this project. Extraction stops if the published checksum does not match.


In [ ]:
def file_digest(path, algorithm, chunk_size=1024 * 1024):
    digest = hashlib.new(algorithm)
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

archive_md5 = file_digest(ARCHIVE_PATH, "md5")
archive_sha256 = file_digest(ARCHIVE_PATH, "sha256")

print("MD5:   ", archive_md5)
print("SHA256:", archive_sha256)
assert archive_md5 == EXPECTED_ARCHIVE_MD5, (
    "Archive checksum mismatch. Delete the file and download it again from the official record."
)

save_json(OUTPUT_DIR / 'archive_verified.json', {
    'archive': str(ARCHIVE_PATH), 'md5': archive_md5, 'sha256': archive_sha256,
})


## 4. Safely extract and locate the dataset root

Archive members are checked against path traversal before extraction.


In [ ]:
def safe_extract(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    # Save completed archive members; incomplete files are never marked complete.
    progress_path = destination / ('.extract_' + archive_path.name + '.json')
    archive_identity = [archive_path.stat().st_size, archive_path.stat().st_mtime_ns]
    progress = json.loads(progress_path.read_text()) if progress_path.exists() else {}
    completed = set(progress.get('completed', [])) if progress.get('archive') == archive_identity else set()
    with ZipFile(archive_path) as archive:
        members = archive.infolist()
        for member in members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f"Unsafe archive member: {member.filename}")
        for index, member in enumerate(members, 1):
            target = destination / member.filename
            if member.filename in completed and (target.is_dir() if member.is_dir() else
                    target.is_file() and target.stat().st_size == member.file_size):
                continue
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                temporary = target.with_name(target.name + '.extracting')
                with archive.open(member) as incoming, temporary.open('wb') as outgoing:
                    shutil.copyfileobj(incoming, outgoing)
                    outgoing.flush()
                    os.fsync(outgoing.fileno())
                temporary.replace(target)
            completed.add(member.filename)
            if index % 100 == 0:
                save_json(progress_path, {'archive': archive_identity, 'completed': sorted(completed)})
                print(f'Extracted {index:,}/{len(members):,}: {archive_path.name}')
        save_json(progress_path, {'archive': archive_identity, 'completed': sorted(completed)})

def find_dataset_root(extract_dir):
    required = [Path("D/CD"), Path("D/UD"), Path("P/CP"), Path("P/UP"), Path("W/CW"), Path("W/UW")]
    direct_candidates = [extract_dir, *[path for path in extract_dir.iterdir() if path.is_dir()]]
    for candidate in direct_candidates:
        if all((candidate / relative).is_dir() for relative in required):
            return candidate
    for cd_dir in extract_dir.rglob("CD"):
        if cd_dir.is_dir():
            candidate = cd_dir.parent.parent
            if all((candidate / relative).is_dir() for relative in required):
                return candidate
    raise FileNotFoundError("Could not find all six SDNET2018 class folders after extraction.")

# Always finish extraction before accepting a root with partially created folders.
safe_extract(ARCHIVE_PATH, EXTRACT_DIR)
inner_archive = next(EXTRACT_DIR.rglob("SDNET2018.zip"), None)
if inner_archive is not None:
    safe_extract(inner_archive, EXTRACT_DIR / "SDNET2018")
DATASET_ROOT = find_dataset_root(EXTRACT_DIR)
save_json(OUTPUT_DIR / 'extraction_complete.json', {'dataset_root': str(DATASET_ROOT)})

print("Dataset root:", DATASET_ROOT.resolve())


## 5. Inventory class folders

Counts are compared with the original publication. A mismatch is reported rather than hidden.


In [ ]:
image_paths_by_code = {}
observed_counts = {}

for code in CLASS_INFO:
    surface_folder = {"D": "D", "P": "P", "W": "W"}[code[-1]]
    class_dir = DATASET_ROOT / surface_folder / code
    paths = sorted(
        path for path in class_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    image_paths_by_code[code] = paths
    observed_counts[code] = len(paths)

print(f"{'Code':<5}{'Observed':>12}{'Expected':>12}{'Status':>12}")
for code in CLASS_INFO:
    status = "OK" if observed_counts[code] == EXPECTED_COUNTS[code] else "MISMATCH"
    print(f"{code:<5}{observed_counts[code]:>12,}{EXPECTED_COUNTS[code]:>12,}{status:>12}")

print(f"Total observed: {sum(observed_counts.values()):,}")
print(f"Total expected: {sum(EXPECTED_COUNTS.values()):,}")
assert sum(observed_counts.values()) >= 56_000

save_json(OUTPUT_DIR / 'inventory.json', observed_counts)


## 6. Audit every image and hash exact content

Files are checked for readability, 256×256 dimensions, RGB-compatible mode, and exact SHA-256 duplicates. Duplicate files remain on disk but all except the first canonical path are marked excluded in the manifest.


In [ ]:
rows = []
paths_by_digest = defaultdict(list)
progress_path = OUTPUT_DIR / ('audit_progress_' + archive_sha256[:16] + '.jsonl')
# Recover complete JSON records; discard only an interrupted trailing write.
if progress_path.exists():
    with progress_path.open('r+b') as stream:
        while True:
            offset = stream.tell()
            line = stream.readline()
            if not line:
                break
            if not line.endswith(b'\n'):
                stream.truncate(offset)
                break
            rows.append(json.loads(line))
completed_paths = {row['path'] for row in rows}
print(f'Resuming audit: {len(rows):,} images saved')
all_paths = [path for code in CLASS_INFO for path in image_paths_by_code[code]]

for index, path in enumerate(all_paths, start=1):
    code = path.parent.name.upper()
    info = CLASS_INFO[code]
    relative_path = path.relative_to(DATASET_ROOT).as_posix()
    if relative_path in completed_paths:
        continue
    readable = True
    width = height = 0
    mode = ""
    error = ""

    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            width, height = image.size
            mode = image.mode
    except Exception as exception:
        readable = False
        error = f"{type(exception).__name__}: {exception}"

    excluded = not readable
    if readable and (width, height) != (256, 256):
        excluded = True
        error = f"unexpected_dimensions:{width}x{height}"
    elif readable and mode != "RGB":
        excluded = True
        error = f"unexpected_mode:{mode}"

    sha256 = file_digest(path, "sha256")
    rows.append({
        "path": relative_path,
        "label": info["label"],
        "class_name": info["class_name"],
        "surface": info["surface"],
        "class_code": code,
        "parent_id": "",
        "sha256": sha256,
        "width": width,
        "height": height,
        "mode": mode,
        "readable": int(readable),
        "excluded": int(excluded),
        "exclusion_reason": error,
        "split": "",
        "split_basis": "unassigned",
    })

    with progress_path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(rows[-1]) + '\n')
        stream.flush()
        os.fsync(stream.fileno())
    if index % 5_000 == 0 or index == len(all_paths):
        print(f"Audited {index:,}/{len(all_paths):,}")

for row in rows:
    paths_by_digest[row['sha256']].append(row['path'])
rows_by_path = {row["path"]: row for row in rows}
duplicate_groups = {
    digest: sorted(paths)
    for digest, paths in paths_by_digest.items()
    if len(paths) > 1
}

for duplicate_paths in duplicate_groups.values():
    canonical = duplicate_paths[0]
    for duplicate in duplicate_paths[1:]:
        row = rows_by_path[duplicate]
        row["excluded"] = 1
        row["exclusion_reason"] = f"exact_duplicate_of:{canonical}"

corrupt_count = sum(not row["readable"] for row in rows)
wrong_size_count = sum(
    row["readable"] and (row["width"], row["height"]) != (256, 256)
    for row in rows
)
non_rgb_count = sum(row["readable"] and row["mode"] != "RGB" for row in rows)
duplicate_file_count = sum(len(paths) - 1 for paths in duplicate_groups.values())

print({
    "corrupt": corrupt_count,
    "wrong_size": wrong_size_count,
    "non_rgb": non_rgb_count,
    "duplicate_groups": len(duplicate_groups),
    "duplicate_files_excluded": duplicate_file_count,
})


## 7. Add verified parent IDs if available

The public class folders do not by themselves prove which tiles came from the same original photograph. To add parent IDs, supply data/parent_mapping.csv with columns path,parent_id, where path matches the manifest-relative path such as D/CD/example.jpg. Otherwise parent_id remains blank.


In [ ]:
PARENT_MAPPING_PATH = DATA_DIR / "parent_mapping.csv"

if PARENT_MAPPING_PATH.exists():
    with PARENT_MAPPING_PATH.open(newline="", encoding="utf-8") as stream:
        parent_mapping = {
            item["path"]: item["parent_id"]
            for item in csv.DictReader(stream)
        }
    unknown_paths = sorted(set(parent_mapping) - set(rows_by_path))
    if unknown_paths:
        raise ValueError(f"Parent mapping contains unknown paths, starting with: {unknown_paths[:3]}")
    for row in rows:
        row["parent_id"] = parent_mapping.get(row["path"], "")
else:
    parent_mapping = {}

eligible_rows = [row for row in rows if not row["excluded"]]
parent_coverage = sum(bool(row["parent_id"]) for row in eligible_rows)
print(f"Verified parent IDs: {parent_coverage:,}/{len(eligible_rows):,}")

if parent_coverage != len(eligible_rows):
    print(
        "Research split is intentionally unassigned: complete verified parent mapping is unavailable."
    )


## 8. Write the audit manifest

This is the complete audit record. It includes excluded files and leaves the research split blank until parent grouping is defensible.


In [ ]:
MANIFEST_FIELDS = [
    "path", "label", "class_name", "surface", "class_code", "parent_id",
    "sha256", "width", "height", "mode", "readable", "excluded",
    "exclusion_reason", "split", "split_basis",
]
AUDIT_MANIFEST_PATH = OUTPUT_DIR / "audit_manifest.csv"

temporary_manifest = AUDIT_MANIFEST_PATH.with_suffix(".csv.tmp")
with temporary_manifest.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=MANIFEST_FIELDS)
    writer.writeheader()
    writer.writerows(sorted(rows, key=lambda row: row["path"]))

temporary_manifest.replace(AUDIT_MANIFEST_PATH)

print("Saved", AUDIT_MANIFEST_PATH.resolve())


## 9. Create an optional preliminary development manifest

The default takes 600 usable images from each class/surface folder, giving 3,600 images balanced across label and surface. Each group is split 80/20. This is a tile-level pipeline check, not a research test.


In [ ]:
DEV_PER_CLASS_SURFACE = 600
DEV_TRAIN_FRACTION = 0.8
development_rows = []

eligible_by_code = defaultdict(list)
for row in eligible_rows:
    eligible_by_code[row["class_code"]].append(row)

for code in CLASS_INFO:
    candidates = sorted(eligible_by_code[code], key=lambda row: row["path"])
    if len(candidates) < DEV_PER_CLASS_SURFACE:
        raise ValueError(f"Not enough usable images in {code}: {len(candidates)}")
    rng = random.Random(SEED + sum(map(ord, code)))
    selected = rng.sample(candidates, DEV_PER_CLASS_SURFACE)
    split_at = round(len(selected) * DEV_TRAIN_FRACTION)
    for index, source_row in enumerate(selected):
        row = dict(source_row)
        row["split"] = "dev_train" if index < split_at else "dev_validation"
        row["split_basis"] = "tile_development_only"
        development_rows.append(row)

DEVELOPMENT_MANIFEST_PATH = OUTPUT_DIR / "development_manifest.csv"
temporary_manifest = DEVELOPMENT_MANIFEST_PATH.with_suffix(".csv.tmp")
with temporary_manifest.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=MANIFEST_FIELDS)
    writer.writeheader()
    writer.writerows(sorted(development_rows, key=lambda row: (row["split"], row["path"])))

temporary_manifest.replace(DEVELOPMENT_MANIFEST_PATH)

dev_counts = Counter(
    (row["split"], row["label"], row["surface"])
    for row in development_rows
)
for key, count in sorted(dev_counts.items()):
    print(key, count)

print("Saved", DEVELOPMENT_MANIFEST_PATH.resolve())
print("WARNING: tile_development_only; possible parent-photo leakage; not a Phase 3 result.")

# One uploadable artifact carries the selected images and manifest into Colab.
BUNDLE_PATH = OUTPUT_DIR / "crackfusion_colab_bundle.zip"
TEMP_BUNDLE_PATH = BUNDLE_PATH.with_suffix(".zip.tmp")
with ZipFile(TEMP_BUNDLE_PATH, "w") as bundle:
    bundle.write(DEVELOPMENT_MANIFEST_PATH, "development_manifest.csv")
    bundle.writestr("bundle_metadata.json", json.dumps({
        "dataset": "SDNET2018",
        "development_images": len(development_rows),
        "split_basis": "tile_development_only",
        "archive_sha256": archive_sha256,
    }, indent=2) + "\n")
    for index, row in enumerate(development_rows, 1):
        bundle.write(DATASET_ROOT / row["path"], f"dataset/SDNET2018/{row['path']}")
        if index % 500 == 0:
            print(f"Packaged {index:,}/{len(development_rows):,} development images")
TEMP_BUNDLE_PATH.replace(BUNDLE_PATH)
print("Colab upload bundle:", BUNDLE_PATH.resolve())


## 10. Visually inspect one usable image per folder

This catches obvious label/folder mistakes before training.


In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(2, 3, figsize=(10, 7))
for axis, code in zip(axes.flat, CLASS_INFO):
    sample_row = next(row for row in eligible_by_code[code] if not row["excluded"])
    with Image.open(DATASET_ROOT / sample_row["path"]) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(f"{code}: {sample_row['class_name']} / {sample_row['surface']}")
    axis.axis("off")

figure.suptitle("SDNET2018 audit samples")
figure.tight_layout()
figure.savefig(OUTPUT_DIR / 'audit_samples.png', dpi=150)
plt.show()


## 11. Save the data-audit summary

The summary separates observed facts from the unresolved parent-lineage gate.


In [ ]:
summary = {
    "stage": "SDNET2018 audit and manifest",
    "official_record": OFFICIAL_RECORD_URL,
    "archive_md5": archive_md5,
    "archive_sha256": archive_sha256,
    "dataset_root": str(DATASET_ROOT.resolve()),
    "expected_counts": EXPECTED_COUNTS,
    "observed_counts": observed_counts,
    "total_images": len(rows),
    "corrupt_images": corrupt_count,
    "wrong_size_images": wrong_size_count,
    "non_rgb_images": non_rgb_count,
    "duplicate_groups": len(duplicate_groups),
    "duplicate_files_excluded": duplicate_file_count,
    "eligible_images": len(eligible_rows),
    "verified_parent_ids": parent_coverage,
    "research_split_status": (
        "parent_mapping_complete" if parent_coverage == len(eligible_rows)
        else "blocked_until_verified_parent_mapping"
    ),
    "development_images": len(development_rows),
    "development_split_basis": "tile_development_only",
    "audit_manifest": str(AUDIT_MANIFEST_PATH.resolve()),
    "development_manifest": str(DEVELOPMENT_MANIFEST_PATH.resolve()),
}

SUMMARY_PATH = OUTPUT_DIR / "data_audit_summary.json"
save_json(SUMMARY_PATH, summary)
print(json.dumps(summary, indent=2))
print("Saved", SUMMARY_PATH.resolve())


## Outputs and training handoff

Audit reports and `crackfusion_colab_bundle.zip` are saved under `MyDrive/Fridge/major_project_dataset/outputs/data_audit`. Notebook 03 loads that bundle automatically. Running the local audit and uploading its finished bundle is faster than extracting thousands of files in Drive.

The 3,600-image split is `tile_development_only`; verified parent mapping and final evaluation remain pending.
